# Apt 305 — every paper result, regenerated end to end on the clean weather file

Produces, in **one run on one weather file**, every number and figure the paper's
results sections need: engine validation against EnergyPlus, the baseline energy
balance, the full ten-state correction trajectory, Tables 4 and 5, and the
diagnostics. **No engine logic changes.** This re-measures the already-validated
engine on correct weather and packages the outputs.

Weather: `AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw` — Essendon
Fields, WMO 958660, tz +10, 8 760 rows, 0 missing wind values, no dead-calm
month, mean 4.84 m/s, 59.8 % of hours above the 4 m/s pivot.

## The consistency rule this notebook exists to enforce

Every ISO result and every EnergyPlus result comes from the **same weather file,
the same building, and the same setpoints** (18 °C heating / 26 °C cooling, read
from the building dictionary and printed). The original validation table was
ISO-on-RO against E+-on-RO; regenerating one side without the other would
recreate exactly the apples-to-oranges error this whole effort untangled. Section
4 asserts each matched input and **aborts** rather than reporting a mismatched
comparison.

## Runtime

| Section | Wall clock |
| --- | --- |
| 1–3 · setup, EnergyPlus, weather | ~4 min |
| 4 · validation + baseline balance | ~4 min |
| 5 · the ten-state trajectory | **~20–25 min** |
| 6 · Tables 4 and 5 | ~8 min |
| 7 · diagnostics | ~5 min |
| 8–9 · suite, index, gate | ~4 min |

**~45–60 min total** on a standard Colab CPU runtime. No GPU. Set
`RERUN_TRAJECTORY = False` in section 1 to reuse the committed trajectory output
and cut ~25 minutes — the numbers are identical either way, since everything
downstream reads that file.

## 1 · Setup

In [ ]:
# Set False to reuse the committed trajectory output instead of re-running the
# ten-state stack (~25 min). Every downstream section reads trajectory_raw.json,
# so the numbers are identical either way — this only decides whether the engine
# is driven again or the committed measurement is trusted.
RERUN_TRAJECTORY = True

import os, subprocess, sys
from pathlib import Path

REPO   = Path('/content/AIB')
BRANCH = 'claude/aib-canonical-clean-weather-0fr0mp'

if not REPO.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/samiraghafarigousheh-sys/aib.git', str(REPO)],
                   check=True)
os.chdir(REPO)

# EVERY branch. The trajectory cherry-picks ten states onto the vendored
# baseline, Table 5b builds worktrees from three historical fix branches, and two
# test modules do the same. Without the full refspec the trajectory cannot
# resolve its commits and those tests SKIP — and a skipped test runs no
# assertions while still leaving exit 0, which is easy to mistake for a pass.
subprocess.run(['git', 'fetch', 'origin', '+refs/heads/*:refs/remotes/origin/*'],
               check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=False)

# Cherry-picking needs a committer identity. A fresh Colab runtime has none and
# git fails with "Please tell me who you are", which points nowhere near the
# harness.
subprocess.run(['git', 'config', 'user.email', 'harness@localhost'], check=True)
subprocess.run(['git', 'config', 'user.name',  'aib-harness'], check=True)

# Colab ships numpy, pandas, matplotlib, plotly, scipy, scikit-learn, pytz,
# requests, tqdm and pytest. These five are the gaps. pyecharts is only a
# rendering library and the engine imports fine without it on this branch — but
# the worktrees of the HISTORICAL branches still import it eagerly, so without it
# Table 5b and two test modules error at fixture setup rather than failing
# honestly.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pvlib', 'timezonefinder', 'holidays', 'workalendar',
                'pyecharts'], check=True)

print(subprocess.run(['git', 'log', '--oneline', '-3'],
                     capture_output=True, text=True).stdout)
print('EPWs present:')
for p in sorted(Path('weather_cache').glob('*.epw')):
    print('  ', p.name)

## 2 · EnergyPlus

Part 1 compares two engines. Without the second there is nothing to report, so
the validation script refuses to run rather than emitting a one-sided table.

In [ ]:
EP = Path('/opt/ep/energyplus')
EP_URL = ('https://github.com/NREL/EnergyPlus/releases/download/v24.1.0/'
          'EnergyPlus-24.1.0-9d7789a3ac-Linux-Ubuntu22.04-x86_64.tar.gz')

if not EP.is_file():
    print('downloading EnergyPlus 24.1.0 (~186 MB)…', flush=True)
    subprocess.run(['wget', '-q', '-O', '/tmp/ep.tar.gz', EP_URL], check=True)
    Path('/opt/ep').mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xzf', '/tmp/ep.tar.gz', '-C', '/opt/ep',
                    '--strip-components=1'], check=True)

print(subprocess.run([str(EP), '--version'], capture_output=True, text=True).stdout.strip())

## 3 · The weather file, screened before anything is computed

Site-correctness was never sufficient. The superseded Melbourne Regional Office
file sat 0.008° from the building — closer than Essendon — and four whole months
of its wind column read exactly 0.0 m/s, because the station's record ends in
2014. `h_ce = 4v + 4` collapses to 4 W/(m²·K) at v = 0 against the ISO constant's
20, so those fabricated calms tripled sensible cooling through C2.

Both files are screened here so the contrast is on the page, then the guard is
demonstrated by pointing it at the file it exists to catch.

In [ ]:
EPW = 'weather_cache/AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw'

subprocess.run([sys.executable, 'tools/diagnostics/weather_integrity.py',
                EPW, 'weather_cache/AUS_VIC_Melbourne.RO.948680_TMYx.2011-2025.epw',
                '--no-assert'])

In [ ]:
sys.path.insert(0, str(REPO / 'examples'))
from weather_melbourne import resolve, CANONICAL_EPW, SUPERSEDED_EPW

print('CANONICAL_EPW  =', CANONICAL_EPW)
print('SUPERSEDED_EPW =', SUPERSEDED_EPW, '\n')

src, path, label = resolve(None, 'auto')          # what the case study resolves to
print('resolved ->', path, '\n')
assert 'Essendon' in path, 'the case study is not on the Essendon file'

print('--- passing the superseded RO file explicitly ---')
try:
    resolve(f'weather_cache/{SUPERSEDED_EPW}', 'epw')
    print('\n!! NOT REFUSED — the guard is not working')
except Exception as exc:
    print('\nREFUSED, correctly:')
    print('   ', str(exc).strip().splitlines()[1].strip())

## 4 · Part 1 — ISO 52016-1 against EnergyPlus, and the baseline balance

Both engines on the same file, the same building, the same setpoints. Ideal
loads, so the comparison is **need vs need**, not system-sized.

Three things this script does that a naive comparison would not, each of which
would otherwise invert or invalidate the result:

1. **The ISO side is the trajectory's Baseline state**, not a separate baseline
   run — the vendored engine with the closure commits cherry-picked on top for
   reporting. It is asserted equal to the trajectory's row 1 before anything is
   written, so the paper's Table 2 and Table 6 row 1 cannot be different numbers
   for one claim.
2. **The neighbour model is probed, not assumed.** The five party surfaces are
   declared `conditioned: True`, and the *baseline engine ignores that* — holding
   conditioned neighbours at setpoint is the Issue-7 fix, trajectory state 7. The
   baseline runs them as ISO 13789 unconditioned buffers, so the IDF reproduces
   the buffer via `OtherSideCoefficients` with `N4 = b_ztu`, `N7 = 1 − b_ztu`,
   using b_ztu values read out of the ISO engine itself.
3. **Internal gains are probed too.** The engine substitutes ISO 16798-1
   tabulated q_int for the building type class and *ignores* the dictionary's
   `full_load` values, so matching the dictionary would not match the engine.

Any input mismatch aborts the run.

In [ ]:
r = subprocess.run([sys.executable, 'tools/paper/validation_iso_vs_ep.py',
                    '--weather', EPW, '--energyplus', str(EP)],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print(r.stderr[-4000:])
print('exit code:', r.returncode, '— non-zero means an input did not match, or the '
      'ISO side did not reproduce the trajectory Baseline')
assert r.returncode == 0

In [ ]:
from IPython.display import display
import json
import pandas as pd

val = pd.read_csv('results/paper/validation_iso_vs_ep/validation_iso_vs_ep.csv')
display(val.style.format({c: '{:,.2f}' for c in val.columns if c != 'Metric'}
                         | {'diff (%)': '{:+.1f} %'}).hide(axis='index'))

h, c, t = (val.iloc[i] for i in range(3))
print(f"heating: ISO {'over' if h['diff (kWh)'] > 0 else 'under'}-predicts by "
      f"{abs(h['diff (kWh)']):,.2f} kWh ({abs(h['diff (%)']):.1f} %)")
print(f"cooling: ISO {'over' if c['diff (kWh)'] > 0 else 'under'}-predicts by "
      f"{abs(c['diff (kWh)']):,.2f} kWh ({abs(c['diff (%)']):.1f} %)")
print(f"total:   {t['diff (kWh)']:+,.2f} kWh ({t['diff (%)']:+.1f} %)")
print()
print('The old table read ISO 15.9 vs E+ 766.2 heating and 2,027.5 vs 900.4 cooling.')
print('The pattern INVERTS — and the cause is the building, not the weather: that')
print('table typed the party surfaces "opaque", which with sky_view_factor 0 the')
print('core maps to GR, giving a third-floor apartment 75.10 m2 of slab-on-ground.')

In [ ]:
from IPython.display import Image, display
display(Image('results/paper/validation_iso_vs_ep/validation_iso_vs_ep.png'))

### The inputs, asserted to match

Read back from what the ISO engine actually loaded and compared with what went
into the IDF.

In [ ]:
from IPython.display import display
raw = json.loads(Path('results/paper/validation_iso_vs_ep/validation_raw.json').read_text())
audit = pd.DataFrame(raw['audit'])
audit.columns = ['Input', 'ISO side', 'EnergyPlus side', 'Match']
display(audit.style.hide(axis='index'))
assert audit['Match'].all(), 'an input did not match'

print('b_ztu, probed out of the ISO engine (NOT recomputed here):')
for z, b in raw['meta']['b_ztu'].items():
    cond = raw['meta']['adjacent_zone_conditioned'].get(z)
    print(f'   {z:<12} b_ztu = {b:.6f}   declared conditioned: {cond}')
print()
print('Internal gains, probed (the engine ignores the dictionary full_load):')
for k, v in raw['meta']['gains_w'].items():
    print(f'   {k:<14} {v:>10,.2f} W')
print()
ov = raw['meta']['overhang']
print(f"Declared window overhang is a no-op on this weather: max |D| = "
      f"{ov['max_abs_diff']:.3e} kWh (the IDF has no shading surface, so this "
      f"had to be verified, not assumed)")

### The baseline energy balance (Part 1d)

In [ ]:
from IPython.display import display, Image
display(Image('results/paper/baseline_balance/baseline_balance_sankey.png'))

bal = json.loads(Path('results/paper/baseline_balance/baseline_balance_raw.json').read_text())
print(f"in {sum(bal['inputs_kWh'].values()):,.1f} / out {sum(bal['outputs_kWh'].values()):,.1f} kWh")
print(f"V2 residual {bal['residual_kWh']:+,.2f} kWh ({bal['residual_pct']:+.2f} % of inputs)")
print(f"transmission line items: {bal['n_transmission_items']}   "
      f"(the old Sankey listed 2 for a seven-surface building)")
print()
print('The residual is drawn as an unmatched gap, on the INPUT side — a negative')
print('residual is unaccounted input, not an extra loss. The old figure republished')
print('it as a "Transmission (residual)" FLOW, which makes any diagram close by')
print('construction and so cannot check closure at all.')

## 5 · Part 2 — the ten-state correction trajectory

Methodology order, literature corrections first, each state the previous one plus
**exactly one** correction cherry-picked onto the unmodified vendored baseline:

    Baseline → +C1 dynamic window → +C2 wind-dependent h_ce → +Ventilation
             → +Latent → +Internal gains → +Conditioned zones → +Ground contact
             → +Hemisphere → +Closure fixes (canonical)

Every state is measured with the **same** closure-capable instrument, which is
the only thing that makes them comparable. An eleventh run measures HEAD directly
as the invariance reference.

Two guardrails, both load-bearing:

* `--closure-base` is **pinned to `978db37`** (the tool's default). PR #16 merged
  three of the four closure commits into `main`, so the old `origin/main` default
  now resolves to a single commit — nine states would be measured with an
  instrument the tenth does not use, the residuals would blow out, and it would
  read as a physics finding. The resolved set is asserted at run start.
* HEAD invariance is checked against a **live HEAD run**, not a stored constant.

In [ ]:
if RERUN_TRAJECTORY:
    proc = subprocess.Popen(
        [sys.executable, 'tools/diagnostics/canonical_trajectory.py',
         '--weather', EPW, '--outdir', 'results/paper/canonical_trajectory',
         '--expect-weather', 'Essendon'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:            # stream, rather than going quiet for 25 min
        print(line, end='')
    proc.wait()
    print('\nexit code:', proc.returncode)
    print('  0 = gate passed, HEAD invariant, engine tree identical, ADJ in '
          'inventory, latent gated')
    print('  2 = a state failed the 5 % V2 residual gate')
    print('  3 = HEAD is not invariant — a correction is not cleanly separable')
    print('  4 = the final engine tree differs from HEAD')
    print('  5 = the transmission inventory or the latent gate failed')
    assert proc.returncode == 0

    subprocess.run([sys.executable, 'tools/diagnostics/make_closed_balance_chart.py',
                    '--raw', 'results/paper/canonical_trajectory/trajectory_raw.json',
                    '--outdir', 'results/paper/canonical_trajectory',
                    '--stem', 'canonical_trajectory',
                    '--title', 'Apt 305, 50 Barry St Carlton — the canonical '
                               'correction trajectory, methodology order',
                    '--note', 'Literature corrections first (C1, C2), then the found '
                              'implementation defects, then the closure fixes. Every '
                              'state measured with the same closure-capable '
                              'instrument. Each metric on its own axis.'], check=True)
else:
    print('RERUN_TRAJECTORY is False — using the committed '
          'results/paper/canonical_trajectory/trajectory_raw.json')
    assert Path('results/paper/canonical_trajectory/trajectory_raw.json').is_file()

In [ ]:
from IPython.display import display
RAW = Path('results/paper/canonical_trajectory/trajectory_raw.json')
blob = json.loads(RAW.read_text())
B = {k: v['config_B'] for k, v in blob['results'].items()}
STATES = list(B)
CANON = STATES[-1]
AREA = float(B[CANON]['net_floor_area_m2'])
WEATHER = Path(blob['meta']['weather']).name

traj = pd.DataFrame([{
    'State': s,
    'Sensible heating (kWh)': B[s]['Q_H_sensible_kWh'],
    'Sensible cooling (kWh)': B[s]['Q_C_sensible_kWh'],
    'Gated latent (kWh)': B[s]['Q_C_latent_kWh'],
    'Ungated latent (kWh)': B[s]['Q_C_latent_ungated_kWh'],
    'Latent heating (kWh)': B[s]['Q_H_latent_kWh'],
    'Sens + gated latent (kWh)': (B[s]['Q_H_sensible_kWh'] + B[s]['Q_C_sensible_kWh']
                                  + B[s]['Q_C_latent_kWh']),
    'Engine total (kWh)': B[s]['Q_need_total_kWh'],
    'kWh/m²·yr': B[s]['Q_need_total_kWh_per_sqm'],
    'V2 residual (kWh)': B[s]['sankey']['residual_kWh'],
    'V2 residual (%)': B[s]['sankey']['residual_pct'],
    'Gate': 'PASS' if abs(B[s]['sankey']['residual_pct']) < 5.0 else 'FAIL',
    'Tr items': B[s]['sankey']['n_transmission_items'],
} for s in STATES])

display(traj.style.format(
    {c: '{:,.2f}' for c in traj.columns if traj[c].dtype.kind == 'f'}
    | {'Latent heating (kWh)': '{:,.4f}', 'V2 residual (kWh)': '{:+,.2f}',
       'V2 residual (%)': '{:+.2f} %'}).hide(axis='index'))

print('`Ungated latent` is a diagnostic contrast column only — the zone moisture')
print('balance before the plant-on gate. It is NEVER part of any total.')
print()
print('`Engine total` additionally carries latent HEATING (~153 kWh of phantom')
print('humidification before the latent fix, 0.00 after), which is why the two')
print('total columns diverge early and coincide at the canonical state.')

In [ ]:
from IPython.display import display, Image
display(Image('results/paper/canonical_trajectory/canonical_trajectory.png'))

### The canonical headline, and the component split

In [ ]:
from IPython.display import display
c = B[CANON]
sens_plus = c['Q_H_sensible_kWh'] + c['Q_C_sensible_kWh'] + c['Q_C_latent_kWh']

print(f'CANONICAL — {WEATHER}\n')
print(f"  {c['Q_H_sensible_kWh']:.2f} kWh sensible heating")
print(f"+ {c['Q_C_sensible_kWh']:.2f} kWh sensible cooling")
print(f"+ {c['Q_C_latent_kWh']:.2f} kWh gated latent")
print(f"= {sens_plus:.2f} kWh  =  {c['Q_need_total_kWh_per_sqm']:.2f} kWh/m²·yr"
      f"   over {AREA:.0f} m²\n")

prior = Path('results/au_canonical/comparison.csv')
if prior.is_file():
    p = pd.read_csv(prior).iloc[-1]
    cmp = pd.DataFrame([
        ('Sensible heating (kWh)', p['Sensible heating (kWh)'], c['Q_H_sensible_kWh']),
        ('Sensible cooling (kWh)', p['Sensible cooling (kWh)'], c['Q_C_sensible_kWh']),
        ('Gated latent (kWh)',     p['Latent cooling, gated (kWh)'], c['Q_C_latent_kWh']),
        ('Total (kWh)',            p['Total (kWh)'], c['Q_need_total_kWh']),
        ('Total (kWh/m²·yr)',      p['Total (kWh/m²)'], c['Q_need_total_kWh_per_sqm']),
    ], columns=['Metric', 'Superseded — RO, corrupt wind', 'This run — Essendon'])
    cmp['Δ']   = cmp['This run — Essendon'] - cmp['Superseded — RO, corrupt wind']
    cmp['Δ %'] = 100 * cmp['Δ'] / cmp['Superseded — RO, corrupt wind']
    display(cmp.style.format({'Superseded — RO, corrupt wind': '{:,.2f}',
                              'This run — Essendon': '{:,.2f}',
                              'Δ': '{:+,.2f}', 'Δ %': '{:+.1f} %'}).hide(axis='index'))

print('QUOTE THE COMPONENTS, NOT THE TOTAL. The total moved -7.1 % while heating')
print('rose 40.9 % and cooling fell 90.6 %, and C2 changed sign. A sentence that')
print('reports only the total describes none of that.')

## 6 · Part 3 — Tables 4 and 5

**Table 4** comes entirely from trajectory states 1–3. Zero extra engine runs, so
the paper's Table 4 and its trajectory rows are the same measurements.

**Table 5 could not.** It is a 2×2 *isolation* experiment and the trajectory is
**cumulative**, so "the latent fix without the ventilation fix" is not a state on
that path — and the six-state closed-balance harness cannot supply it either, its
second state having both fixes already combined. So both forms are emitted:

* **5a** — methodology order, derived from the trajectory, zero extra runs.
* **5b** — the 2×2, four states from the same vendored baseline on the same
  instrument. Its `Base` is reconciled against the trajectory's `Baseline` before
  anything is written; if they differ, the run stops.

They answer different questions and their columns are not interchangeable.

In [ ]:
r = subprocess.run([sys.executable, 'tools/paper/tables_4_5.py', '--weather', EPW],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
assert r.returncode == 0

In [ ]:
from IPython.display import display
for name, path in [('Table 4 — window / h_ce',
                    'results/paper/tables_4_5/table4_window_hce.csv'),
                   ('Table 5a — methodology order',
                    'results/paper/tables_4_5/table5a_methodology_order.csv'),
                   ('Table 5b — isolated 2×2',
                    'results/paper/tables_4_5/table5b_isolation_2x2.csv')]:
    print(f'\n=== {name} ===')
    df = pd.read_csv(path)
    display(df.style.format({c: '{:,.2f}' for c in df.columns
                             if df[c].dtype.kind == 'f'}).hide(axis='index'))

t5b = json.loads(Path('results/paper/tables_4_5/table5b_raw.json').read_text())
s = t5b['summary']
print(f"\n2x2 reconciliation against the trajectory Baseline: "
      f"max |Δ| {s['base_reconciliation_max_abs_kWh']:.1e} kWh")
print(f"Latent-cooling interaction term: {s['latent_interaction_kWh']:+.2f} kWh "
      f"— non-zero, so the two fixes are NOT additive on the latent side, which is "
      f"the claim the 2x2 exists to make.")
print(f"Sensible side separates cleanly: {s['sensible_separates']}")

### Which table came from where

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/paper/tables_4_5/PROVENANCE.md').read_text()))

## 7 · Part 4 — the diagnostics

Wind, Sankey decomposition, the latent breakdown, and the residual across all ten
states. Everything except the wind diagnostic is read out of the trajectory's raw
output — no re-measurement, so the figures cannot disagree with the tables.

In [ ]:
# 7a — wind. Two engine runs (dynamic h_ce vs the ISO constant), one switch apart.
proc = subprocess.Popen(
    [sys.executable, 'tools/diagnostics/wind_h_ce_diagnostic.py',
     '--weather', EPW, '--outdir', 'results/paper/diagnostics/wind',
     '--tag', 'essendon', '--expect-weather', 'Essendon',
     '--compare-to', 'results/diagnostics/wind_stats.json'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    if 'it/s' in line or 'it]' in line or line.startswith('[t='):
        continue                       # tqdm bars and per-timestep traces
    print(line, end='')
proc.wait()
assert proc.returncode == 0

In [ ]:
from IPython.display import display, Image
display(Image('results/paper/diagnostics/wind/wind_distribution_essendon.png'))

w = json.loads(Path('results/paper/diagnostics/wind/wind_stats_essendon.json').read_text())
s = w['summary']
print(f"VERDICT ({w['verdict']}) — replaces the earlier (c) reached on the RO file\n")
print(f"  C2 moves sensible cooling {s['delta_C']:+.2f} kWh "
      f"({s['C_fix']:.2f} → {s['C_dyn']:.2f})")
print(f"  {s['pct_extra_from_nonzero_wind']:.1f} % of that from genuine, non-zero "
      f"wind bands  (was 4 % on the RO file)")
print(f"  {s['pct_cooling_hours_above_pivot']:.1f} % of the {s['n_cooling_dyn']} "
      f"cooling-plant hours sit above the 4 m/s pivot, against "
      f"{s['pct_hours_above_pivot']:.1f} % of the year")
print()
print('C2 now REDUCES cooling — the opposite sign to the RO artefact. With 59.8 %')
print('of hours above the pivot, 4v+4 sits ABOVE the ISO fixed 20 W/(m2.K) for most')
print('of the year instead of collapsing to a fifth of it, so a stronger external')
print('film sheds more absorbed solar from the west wall back to the air.')

In [ ]:
# 7b — Sankey for the states the paper narrates; 7c/7d — latent and residual.
subprocess.run([sys.executable, 'tools/paper/baseline_balance.py',
                '--raw', 'results/paper/canonical_trajectory/trajectory_raw.json',
                '--state', 'Baseline', '--stem', 'baseline',
                '--state', '+C2 wind-dependent h_ce', '--stem', 'c2_wind_hce',
                '--state', '+Conditioned zones', '--stem', 'conditioned_zones',
                '--state', '+Closure fixes', '--stem', 'canonical',
                '--outdir', 'results/paper/diagnostics/sankey'], check=True)

for what, out in (('latent', 'results/paper/diagnostics/latent'),
                  ('residual', 'results/paper/diagnostics/residual')):
    subprocess.run([sys.executable, 'tools/paper/diagnostics_latent_residual.py',
                    '--what', what,
                    '--raw', 'results/paper/canonical_trajectory/trajectory_raw.json',
                    '--outdir', out], check=True)

In [ ]:
from IPython.display import display, Image
for p in ['results/paper/diagnostics/sankey/baseline_sankey.png',
          'results/paper/diagnostics/sankey/canonical_sankey.png',
          'results/paper/diagnostics/latent/latent_breakdown.png',
          'results/paper/diagnostics/residual/residual_by_state.png']:
    display(Image(p))

## 8 · The regression suite

Must be **green, exit 0, nothing skipped**. A skipped worktree test executed no
assertions while still leaving exit 0 — the full refspec fetched in section 1 is
what keeps those from skipping, and the assertion below is what catches it if
they do.

In [ ]:
out = Path('results/paper/pytest.txt')
env = dict(os.environ, PYTHONPATH=str(REPO / 'pybuildingenergy' / 'src'))
proc = subprocess.Popen([sys.executable, '-m', 'pytest', 'tests/', '-v', '-rs',
                         '--tb=short', '-p', 'no:cacheprovider'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
lines = []
for line in proc.stdout:
    lines.append(line)
    if any(k in line for k in ('FAILED', 'ERROR', 'SKIPPED', '=====')):
        print(line, end='')
proc.wait()
lines.append(f'\nexit code: {proc.returncode}\n')
out.write_text(''.join(lines))
print(''.join(lines[-3:]))

assert proc.returncode == 0, 'the regression suite is not green'
assert not any('SKIPPED' in l for l in lines), \
    'a test skipped — it executed no assertions; check the branch fetch in section 1'

## 9 · The gate, and the index

`build_index.py` **enforces** the gate rather than describing it: it reads every
artefact back, checks all seven conditions, writes `INDEX.md` with the result at
the top, and exits non-zero if any fails. No number in the index is retyped —
an index that restates a figure by hand is one more place for the paper to
disagree with the run that produced it.

In [ ]:
r = subprocess.run([sys.executable, 'tools/paper/build_index.py'],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode != 0 else '')
assert r.returncode == 0, ('THE GATE FAILED — no reduction percentage and no '
                           'kWh/m² headline from this run is final. See INDEX.md.')

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/paper/INDEX.md').read_text()))

### What the paper text must be redone against

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/paper/SUPERSEDED.md').read_text()))

## 10 · Before quoting any of this

1. **The gate is the whole point.** No reduction percentage and no kWh/m²
   headline is final unless the V2 residual is under 5 % on **every** state, the
   ADJ transmission is in the inventory, the latent is gated, and the suite is
   green. Section 9 raises if any of that fails; nothing above it is quotable
   from a run where it did.

2. **Quote the components, not the total.** The canonical total moved −7.1 %
   against the superseded run while heating rose 40.9 % and cooling fell 90.6 %.
   The same applies to the validation table: its total (+33.2 %) is worse than
   the cooling error and better than the heating error, and summarises neither.

3. **The validation pattern inverted, and the weather is not why.** The old
   "ISO under-predicts heating, over-predicts cooling" was a property of a
   *mis-specified building* — party surfaces typed `opaque`, hence buried as
   slab-on-ground. That input is worth thousands of kWh; the weather change is
   worth tens. Do not attribute the inversion to Essendon.

4. **C2 changed sign.** Any text describing the wind-dependent h_ce as
   *increasing* cooling is now wrong, not merely imprecise.

5. **Part 1 validates the BASELINE engine**, before any of the nine corrections.
   Its heating disagreement is the *starting* discrepancy the trajectory then
   addresses — not the paper's final accuracy claim.

6. **Table 5a and 5b are not interchangeable.** 5a answers "what does each fix
   add where the methodology applies it"; 5b answers "are the two fixes
   separable". Cite deliberately — see `tables_4_5/PROVENANCE.md`.

7. **The RO file stays in `weather_cache/` deliberately.** The before/after wind
   contrast needs it; `CANONICAL_EPW` and the wind preflight are what stop it
   being picked up by accident. Do not "tidy" either away.